In [1]:
"""
Module: whisper_studio.py
Description: Professional AI Transcription Studio using PyQt6 and OpenAI Whisper.
Technical Note: Implements QThread to prevent GUI blocking during inference.
"""

import sys
import os
import whisper
from PyQt6.QtWidgets import (QApplication, QMainWindow, QPushButton, QVBoxLayout, 
                             QHBoxLayout, QWidget, QLabel, QFileDialog, QTextEdit, QFrame, QComboBox)
from PyQt6.QtCore import Qt, QThread, pyqtSignal
from googletrans import Translator

class TranscriptionWorker(QThread):
    """
    Worker thread to handle the heavy AI transcription task.
    Using a separate thread ensures the GUI remains responsive.
    """
    # Signals to communicate back to the Main Window
    finished = pyqtSignal(str)  # Sends the final text result
    error = pyqtSignal(str)     # Sends any error message

    def __init__(self, model, file_path):
        super().__init__()
        self.model = model
        self.file_path = file_path

    def run(self):
        """The actual code that runs in the background."""
        try:
            # Execute the transcription
            result = self.model.transcribe(self.file_path)
            # Emit the success signal with the text
            self.finished.emit(result["text"].strip())
        except Exception as e:
            # Emit the error signal if something fails
            self.error.emit(str(e))

class WhisperApp(QMainWindow):
    def __init__(self):
        super().__init__()

        # --- AI Initialization ---
        # Load model once during startup to save time later
        print("Loading Whisper 'base' model...")
        self.model = whisper.load_model("base")
        self.current_file = None

        # --- UI Setup ---
        self.setWindowTitle("AI Lyrics Studio")
        self.setMinimumSize(900, 600)
        
        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        self.main_layout = QHBoxLayout(self.central_widget)

        # --- Sidebar & Controls ---
        self.sidebar = QFrame()
        self.sidebar.setFixedWidth(250)
        self.sidebar.setStyleSheet("background-color: #2c3e50; border-radius: 10px;")
        self.sidebar_layout = QVBoxLayout(self.sidebar)

        self.btn_import = QPushButton("📁 Import Audio")
        self.btn_transcribe = QPushButton("⚡ Start AI")
        self.btn_transcribe.setEnabled(False) # Start disabled
        self.btn_clear = QPushButton("🗑️ Clear Workspace")

        # 1. Create the Language Label and Dropdown
        self.lang_label = QLabel("Target Language:")
        self.lang_label.setStyleSheet("color: white; margin-top: 10px;")
        
        self.combo_lang = QComboBox()
        self.combo_lang.addItems(["English", "French", "Spanish", "German", "Italian"])
        self.combo_lang.setStyleSheet("padding: 5px; background-color: white;")

        # 2. Add them to the sidebar layout
        self.sidebar_layout.addWidget(self.lang_label)
        self.sidebar_layout.addWidget(self.combo_lang)

        # Professional Styling
        btn_css = "QPushButton { background-color: #34495e; color: white; padding: 12px; border: none; text-align: left; }"
        self.btn_import.setStyleSheet(btn_css)
        self.btn_transcribe.setStyleSheet(btn_css + "QPushButton:enabled { background-color: #27ae60; }")
        self.btn_clear.setStyleSheet(btn_css)

        self.sidebar_layout.addWidget(self.btn_import)
        self.sidebar_layout.addWidget(self.btn_transcribe)
        self.sidebar_layout.addWidget(self.btn_clear)
        self.sidebar_layout.addStretch()

        # --- Workspace ---
        self.workspace = QVBoxLayout()
        self.status_label = QLabel("Ready. Please import a file.")
        self.text_editor = QTextEdit()
        self.text_editor.setPlaceholderText("Lyrics will appear here...")
        
        self.workspace.addWidget(self.status_label)
        self.workspace.addWidget(self.text_editor)

        self.main_layout.addWidget(self.sidebar)
        self.main_layout.addLayout(self.workspace)

        # --- Connections ---
        self.btn_import.clicked.connect(self.select_audio)
        self.btn_transcribe.clicked.connect(self.start_task)
        self.btn_clear.clicked.connect(self.clear_workspace)

        self.translator = Translator()

        self.lang_map = {
            "English": "en",
            "French": "fr",
            "Spanish": "es",
            "German": "de",
            "Italian": "it"
        }

    def select_audio(self):
        path, _ = QFileDialog.getOpenFileName(self, "Open Audio", "", "Audio (*.mp3 *.wav)")
        if path:
            self.current_file = path
            self.status_label.setText(f"Loaded: {os.path.basename(path)}")
            self.btn_transcribe.setEnabled(True)

    def start_task(self):
        """Initializes and starts the background worker thread."""
        self.status_label.setText("AI is transcribing... (The window is still active!)")
        self.btn_transcribe.setEnabled(False) # Prevent double clicking
        
        # Create and start the thread
        self.worker = TranscriptionWorker(self.model, self.current_file)
        self.worker.finished.connect(self.handle_result)
        self.worker.error.connect(self.handle_error)
        self.worker.start()

    def handle_result(self, text):
        """Processes the transcription and translates if necessary."""
        target_lang_name = self.combo_lang.currentText()
        target_code = self.lang_map.get(target_lang_name, "en")

        # If user wants translation (and it's not already English)
        if target_code != "en":
            self.status_label.setText(f"Translating to {target_lang_name}...")
            try:
                # Perform the translation
                translated = self.translator.translate(text, dest=target_code)
                self.text_editor.setPlainText(translated.text)
                self.status_label.setText(f"Translation to {target_lang_name} Complete!")
            except Exception as e:
                self.text_editor.setPlainText(f"Transcription was successful, but translation failed: {text}")
                self.status_label.setText("Translation Error.")
        else:
            # Standard English output
            self.text_editor.setPlainText(text)
            self.status_label.setText("Transcription Complete!")
        
        self.btn_transcribe.setEnabled(True)
        self.btn_save.setEnabled(True)

    def handle_error(self, message):
        """Called if the AI fails."""
        self.status_label.setText(f"Error: {message}")
        self.btn_transcribe.setEnabled(True)
        
    def clear_workspace(self):
        """Resets the text editor and status label."""
        self.text_editor.clear()
        self.status_label.setText("Workspace cleared. Ready for new file.")


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = WhisperApp()
    window.show()
    sys.exit(app.exec())

Loading Whisper 'base' model...


C:\Users\assyl\anaconda3\envs\whisper_app\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


AttributeError: 'WhisperApp' object has no attribute 'btn_save'

AttributeError: 'WhisperApp' object has no attribute 'btn_save'

SystemExit: 0

C:\Users\assyl\anaconda3\envs\whisper_app\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
